# BAA10Y Adaptive Agent Test

This adaptive agent check the following two questions

1. **Does the adaptive-agent workflow work technically?**
2. **Does a tested parameter candidate improve CRPS versus its baseline?**

The default test runs two LinearRegression candidates on the short `smoke` window. LLMP is disabled by default because it uses proxy calls. The protected 2026 evaluation is never used for tuning.

In [13]:
print("hello")

hello


In [14]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')


import pandas as pd
from IPython.display import display


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'aieng-forecasting').is_dir():
            return candidate
    raise RuntimeError('Could not find repository root.')


CHECKS = []


def add_check(category: str, name: str, passed: bool, details: str = '') -> None:
    CHECKS.append(
        {
            'category': category,
            'check': name,
            'status': 'PASS' if passed else 'NOT YET',
            'details': details,
        }
    )


ROOT = find_repo_root()
STATE_PATH = (
    ROOT
    / 'implementations'
    / 'BAA10Y_forecasting'
    / 'adaptive_agent'
    / 'state'
    / 'tuning_state.yaml'
)

add_check('technical', 'Repository root found', ROOT.exists(), str(ROOT))
print('STATE_PATH:', STATE_PATH)

STATE_PATH: /home/coder/agentic-forecasting/implementations/BAA10Y_forecasting/adaptive_agent/state/tuning_state.yaml


## 1. Verify modules and tools

In [15]:
from BAA10Y_forecasting.adaptive_agent.agent import build_baa10y_adaptive_config
from BAA10Y_forecasting.adaptive_agent.skill_state import TuningStateStore
from BAA10Y_forecasting.adaptive_agent.skill_tools import build_baa10y_tuning_tools
from BAA10Y_forecasting.adaptive_agent.tuner import BAA10YTuner

add_check('technical', 'Adaptive modules import', True)

TOOLS = build_baa10y_tuning_tools(state_path=STATE_PATH)
TOOL_BY_NAME = {tool.__name__: tool for tool in TOOLS}

EXPECTED_TOOLS = {
    'get_tuning_state',
    'list_tuning_candidates',
    'run_tuning_trial',
    'compare_tuning_trials',
}

tool_names = set(TOOL_BY_NAME)
tools_ok = EXPECTED_TOOLS.issubset(tool_names)
add_check('technical', 'Four tuning tools registered', tools_ok, str(sorted(tool_names)))

print('Connected tools:')
for name in sorted(tool_names):
    print(' -', name)

assert tools_ok, f'Missing tools: {sorted(EXPECTED_TOOLS - tool_names)}'

Connected tools:
 - compare_tuning_trials
 - get_tuning_state
 - list_tuning_candidates
 - run_tuning_trial


## 2. Verify candidate registry

In [16]:
candidate_output = TOOL_BY_NAME['list_tuning_candidates']()
candidates = json.loads(candidate_output)
candidate_df = pd.DataFrame(candidates)

required_methods = {
    'linear_regression',
    'lightgbm',
    'llmp_sampled_trajectory',
}

methods_found = set(candidate_df['method'])
registry_ok = required_methods.issubset(methods_found)
baseline_ok = candidate_df.groupby('method')['is_baseline'].any().all()

add_check('technical', 'All three predictor families registered', registry_ok, str(sorted(methods_found)))
add_check('technical', 'Each family has a baseline', bool(baseline_ok))

display(
    candidate_df[
        [
            'candidate_id',
            'method',
            'description',
            'covariate_panel',
            'is_baseline',
            'params',
        ]
    ]
)

assert registry_ok
assert baseline_ok

,candidate_id,method,description,covariate_panel,is_baseline,params
0,linreg_l5_target,linear_regression,Baseline: 5 target lags,target_only,True,"{'lags': 5, 'num_samples': 100}"
1,linreg_l10_target,linear_regression,10 target lags,target_only,False,"{'lags': 10, 'num_samples': 100}"
2,linreg_l5_cov,linear_regression,5 target and covariate lags,default,False,"{'lags': 5, 'lags_past_covariates': 5, 'num_sa..."
3,lightgbm_l5_target,lightgbm,Baseline: 5 target lags,target_only,True,"{'lags': 5, 'num_samples': 100, 'lgbm_kwargs':..."
4,lightgbm_l5_cov,lightgbm,5 target and covariate lags,default,False,"{'lags': 5, 'lags_past_covariates': 5, 'num_sa..."
5,lightgbm_l10_cov_tuned,lightgbm,Tuned LightGBM with 10 lags and covariates,default,False,"{'lags': 10, 'lags_past_covariates': 10, 'num_..."
6,llmp_n8_h48_target,llmp_sampled_trajectory,Baseline: 8 samples and 48-day history,target_only,True,"{'n_samples': 8, 'history_window': 48}"
7,llmp_n12_h48_target,llmp_sampled_trajectory,Increase trajectory samples from 8 to 12,target_only,False,"{'n_samples': 12, 'history_window': 48}"
8,llmp_n8_h64_target,llmp_sampled_trajectory,Increase history window from 48 to 64,target_only,False,"{'n_samples': 8, 'history_window': 64}"
9,llmp_n8_h48_cov,llmp_sampled_trajectory,Baseline LLMP parameters with covariates,default,False,"{'n_samples': 8, 'history_window': 48}"


## 3. Choose the test

Start with the LinearRegression smoke test. The other tests can be enabled later.

In [17]:
HORIZON = 5
EXPERIMENT = 'smoke'
FORCE_REFRESH = False

RUN_LINEAR_TEST = False
RUN_LIGHTGBM_TEST = False
RUN_LLMP_TEST = True  # Keep False until the numerical tests pass.

LINEAR_CANDIDATES = ['linreg_l5_target', 'linreg_l10_target']
LIGHTGBM_CANDIDATES = ['lightgbm_l5_target', 'lightgbm_l5_cov']
LLMP_CANDIDATES = ['llmp_n8_h48_target', 'llmp_n12_h48_target']

print(f'Test window: {EXPERIMENT}; horizon: {HORIZON} business days')

Test window: smoke; horizon: 5 business days


In [18]:
def call_json_tool(tool_name: str, **kwargs):
    output = TOOL_BY_NAME[tool_name](**kwargs)
    try:
        return json.loads(output)
    except json.JSONDecodeError as exc:
        raise RuntimeError(output) from exc


def run_pair(method: str, candidate_ids: list[str]) -> pd.DataFrame:
    for candidate_id in candidate_ids:
        result = call_json_tool(
            'run_tuning_trial',
            candidate_id=candidate_id,
            horizon=HORIZON,
            experiment=EXPERIMENT,
            force_refresh=FORCE_REFRESH,
        )
        print(candidate_id, '->', result['status'], 'CRPS =', result['trial']['mean_crps'])

    comparison = call_json_tool(
        'compare_tuning_trials',
        method=method,
        horizon=HORIZON,
        experiment=EXPERIMENT,
    )

    raw = pd.DataFrame(comparison['results'])
    summary = (
        raw.groupby(['candidate_id', 'is_baseline'], as_index=False)
        .agg(mean_crps=('mean_crps', 'mean'), runs=('mean_crps', 'size'))
        .sort_values('mean_crps')
        .reset_index(drop=True)
    )

    baseline = summary.loc[summary['is_baseline'], 'mean_crps']
    if not baseline.empty:
        baseline_crps = float(baseline.iloc[0])
        summary['improvement_pct'] = 100.0 * (baseline_crps - summary['mean_crps']) / baseline_crps

    display(summary)
    return summary

## 4. Run smoke tests

In [19]:
TEST_RESULTS = {}

if RUN_LINEAR_TEST:
    TEST_RESULTS['linear_regression'] = run_pair('linear_regression', LINEAR_CANDIDATES)

if RUN_LIGHTGBM_TEST:
    TEST_RESULTS['lightgbm'] = run_pair('lightgbm', LIGHTGBM_CANDIDATES)

if RUN_LLMP_TEST:
    if EXPERIMENT == 'stress_2020':
        raise ValueError('LLMP cannot be run on stress_2020.')
    TEST_RESULTS['llmp_sampled_trajectory'] = run_pair(
        'llmp_sampled_trajectory',
        LLMP_CANDIDATES,
    )

tests_executed = bool(TEST_RESULTS)
add_check('technical', 'At least one candidate pair executed', tests_executed)

for method, table in TEST_RESULTS.items():
    best = table.iloc[0]
    improvement = float(best.get('improvement_pct', 0.0))
    add_check(
        'performance',
        f'{method} candidate beats smoke baseline',
        improvement > 0,
        f"best={best['candidate_id']}; improvement={improvement:.2f}%",
    )

llmp_n8_h48_target -> saved CRPS = 4.792520661157029
llmp_n12_h48_target -> saved CRPS = 4.5370316804407755


,candidate_id,is_baseline,mean_crps,runs,improvement_pct
0,llmp_n12_h48_target,False,4.537032,1,5.330994
1,llmp_n8_h48_target,True,4.792521,1,0.000000


## 5. Verify adaptive memory

The agent should retain trials after the notebook or ADK session restarts.

In [20]:
first_load = TuningStateStore(STATE_PATH).load()
second_load = TuningStateStore(STATE_PATH).load()

state_ok = (
    STATE_PATH.exists()
    and len(first_load.trials) == len(second_load.trials)
    and len(first_load.trials) > 0
)

add_check(
    'technical',
    'Trial state persists across reloads',
    state_ok,
    f'saved_trials={len(first_load.trials)}',
)

state_df = pd.DataFrame([trial.model_dump(mode='json') for trial in first_load.trials])
if not state_df.empty:
    display(
        state_df[
            [
                'candidate_id',
                'method',
                'horizon',
                'experiment',
                'mean_crps',
                'ran_at',
            ]
        ].sort_values(['method', 'horizon', 'mean_crps'])
    )

,candidate_id,method,horizon,experiment,mean_crps,ran_at
2,lightgbm_l5_target,lightgbm,5,smoke,3.620545,2026-08-20T19:42:35.838392
3,lightgbm_l5_cov,lightgbm,5,smoke,4.260041,2026-08-20T19:45:21.650863
1,linreg_l10_target,linear_regression,5,smoke,3.443620,2026-08-20T19:42:12.812518
0,linreg_l5_target,linear_regression,5,smoke,3.634654,2026-08-20T19:41:08.838307
5,llmp_n12_h48_target,llmp_sampled_trajectory,5,smoke,4.537032,2026-08-20T20:05:26.187254
4,llmp_n8_h48_target,llmp_sampled_trajectory,5,smoke,4.792521,2026-08-20T20:05:19.386054


## 6. Verify ADK configuration

In [21]:
agent_config = build_baa10y_adaptive_config(state_path=STATE_PATH)
config_tool_names = {tool.__name__ for tool in agent_config.extra_tools}
config_ok = EXPECTED_TOOLS.issubset(config_tool_names)

add_check('technical', 'ADK config contains tuning tools', config_ok, str(sorted(config_tool_names)))

print('Agent name:', agent_config.name)
print('Model:', agent_config.model)
print('Tools:', sorted(config_tool_names))

assert config_ok

Agent name: baa10y_adaptive_tuner
Model: gemini-3.1-flash-lite-preview
Tools: ['compare_tuning_trials', 'get_tuning_state', 'list_tuning_candidates', 'run_tuning_trial']


## 7. Final health report

In [22]:
health_report = pd.DataFrame(CHECKS)
display(health_report)

technical = health_report[health_report['category'] == 'technical']
technical_ok = bool((technical['status'] == 'PASS').all())

if technical_ok:
    print('TECHNICAL RESULT: PASS — the adaptive tuning workflow is connected and persistent.')
else:
    print('TECHNICAL RESULT: NOT READY — review the failed technical checks above.')

performance = health_report[health_report['category'] == 'performance']
if not performance.empty and (performance['status'] == 'PASS').any():
    print('PERFORMANCE RESULT: At least one candidate beat its smoke baseline.')
    print('Next: confirm the result with EXPERIMENT = backtest_2025.')
elif not performance.empty:
    print('PERFORMANCE RESULT: No tested candidate beat its smoke baseline.')
    print('The agent still works technically; try another candidate configuration.')


,category,check,status,details
0,technical,Repository root found,PASS,/home/coder/agentic-forecasting
1,technical,Adaptive modules import,PASS,
2,technical,Four tuning tools registered,PASS,"['compare_tuning_trials', 'get_tuning_state', ..."
3,technical,All three predictor families registered,PASS,"['lightgbm', 'linear_regression', 'llmp_sample..."
4,technical,Each family has a baseline,PASS,
5,technical,At least one candidate pair executed,PASS,
6,performance,llmp_sampled_trajectory candidate beats smoke ...,PASS,best=llmp_n12_h48_target; improvement=5.33%
7,technical,Trial state persists across reloads,PASS,saved_trials=6
8,technical,ADK config contains tuning tools,PASS,"['compare_tuning_trials', 'get_tuning_state', ..."


TECHNICAL RESULT: PASS — the adaptive tuning workflow is connected and persistent.
PERFORMANCE RESULT: At least one candidate beat its smoke baseline.
Next: confirm the result with EXPERIMENT = backtest_2025.


## 8. Interactive ADK test

From `implementations/BAA10Y_forecasting`, run:

```bash
uv run adk web .
```

Select the adaptive agent and ask these questions in order:

1. `List the LinearRegression candidates for horizon 5.`
2. `Run linreg_l5_target on the smoke window for horizon 5.`
3. `Run linreg_l10_target on the smoke window for horizon 5.`
4. `Compare the LinearRegression smoke results for horizon 5.`

The conversational agent works correctly if it calls the tools, reports the same CRPS values as this notebook, and remembers the trials after restarting ADK.

> A smoke winner is only a screening result. Confirm improvement on `backtest_2025`; use `stress_2020` only for numerical models; never tune on `eval_2026`.